# Test 8: Qualitative Error Analysis of False Negatives

This notebook isolates the False Negatives (actual malware predicted as safe) from the GraphCodeBERT validation set, and prints the raw source code for the top 10 most confident mistakes.

**Goal**: Identify structural patterns in missed vulnerabilities (e.g., extremely short functions, specific CWEs, or SQL injections) to provide transparent qualitative analysis for the paper.

**Kaggle Environment Details**:
- Dataset: `/kaggle/input/datasets/hasanmahmudabdullah/dfgdataset2/dataset_graphcodebert.jsonl`
- Model: `/kaggle/input/datasets/hasanmahmudabdullah/model2/model.bin`

In [1]:
import torch
import json
import numpy as np
from transformers import AutoTokenizer, RobertaConfig, RobertaModel
from torch.utils.data import DataLoader, SequentialSampler, Dataset, random_split
from tqdm.auto import tqdm
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import CrossEntropyLoss

print("Imports complete.")

Imports complete.


In [2]:
class Args:
    code_length = 384
    data_flow_length = 128
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    eval_batch_size = 32
    seed = 42
    
args = Args()

class Model(nn.Module):   
    def __init__(self, encoder, config, tokenizer, args):
        super(Model, self).__init__()
        self.encoder = encoder
        self.config = config
        self.tokenizer = tokenizer
        self.args = args
        
        # Classification Head
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, 2) # 0 or 1

    def forward(self, input_ids=None, p_ids=None, attn_mask=None, labels=None): 
        if input_ids is not None:
            extended_attention_mask = (1.0 - attn_mask) * -10000.0
            extended_attention_mask = extended_attention_mask.unsqueeze(1)
            
            embedding_output = self.encoder.embeddings(
                input_ids=input_ids, 
                position_ids=p_ids
            )

            encoder_outputs = self.encoder.encoder(
                embedding_output,
                attention_mask=extended_attention_mask,
                head_mask=[None] * self.config.num_hidden_layers
            )
            
            sequence_output = encoder_outputs[0]
            logits = self.classifier(self.dropout(sequence_output[:, 0, :]))
            prob = F.softmax(logits, dim=-1)

            if labels is not None:
                loss_fct = CrossEntropyLoss()
                loss = loss_fct(logits, labels)
                return loss, prob
            else:
                return prob

In [3]:
class TextDataset(Dataset):
    def __init__(self, tokenizer, args, file_path):
        self.args = args
        self.tokenizer = tokenizer
        self.total_len = args.code_length + args.data_flow_length
        
        with open(file_path, 'r') as f:
            self.lines = f.readlines()
            
    def __len__(self):
        return len(self.lines)

    def _get_char_index(self, code_lines, coord):
        row, col = coord
        char_idx = 0
        for i in range(min(row, len(code_lines))):
            char_idx += len(code_lines[i]) 
        return char_idx + col

    def __getitem__(self, item):
        line = self.lines[item]
        # Store the raw index for easy lookup later
        self.last_item_idx = item
        
        entry = json.loads(line)
        code = entry.get('code', '')
        dfg = entry.get('dfg', [])[:self.args.data_flow_length]
        label = int(entry.get('label', 0)) if entry.get('label') is not None else 0

        tokens_obj = self.tokenizer(
            code, 
            max_length=self.args.code_length, 
            truncation=True, 
            padding='max_length',
            return_offsets_mapping=True
        )
        input_ids = tokens_obj['input_ids']
        offsets = tokens_obj['offset_mapping']
        code_lines = code.splitlines(keepends=True)

        dfg_ids = [self.tokenizer.unk_token_id] * len(dfg)
        node_to_token_map = {}
        pos_to_node_idx = {}

        for node_idx, node_item in enumerate(dfg):
            start_pos = node_item[1][0]
            end_pos = node_item[1][1]
            pos_key = (start_pos[0], start_pos[1], end_pos[0], end_pos[1])
            pos_to_node_idx[pos_key] = node_idx
            
            try:
                char_start = self._get_char_index(code_lines, start_pos)
                char_end = self._get_char_index(code_lines, end_pos)
                aligned_tokens = []
                for t_idx, (t_start, t_end) in enumerate(offsets):
                    if t_start == t_end: continue
                    if (t_start >= char_start and t_end <= char_end) or \
                       (char_start >= t_start and char_end <= t_end):
                        aligned_tokens.append(t_idx)
                node_to_token_map[node_idx] = aligned_tokens
            except IndexError:
                node_to_token_map[node_idx] = []

        total_len = self.args.code_length + len(dfg)
        attn_mask = np.zeros((self.total_len, self.total_len), dtype=bool)
        c_len = self.args.code_length
        attn_mask[:c_len, :c_len] = True
        
        for node_idx, node_item in enumerate(dfg):
            abs_node_idx = c_len + node_idx
            tokens = node_to_token_map.get(node_idx, [])
            for t_idx in tokens:
                attn_mask[abs_node_idx, t_idx] = True
                attn_mask[t_idx, abs_node_idx] = True
            
            parent_positions = node_item[4]
            for p_pos in parent_positions:
                p_key = (p_pos[0][0], p_pos[0][1], p_pos[1][0], p_pos[1][1])
                if p_key in pos_to_node_idx:
                    parent_idx = pos_to_node_idx[p_key]
                    abs_parent_idx = c_len + parent_idx
                    attn_mask[abs_node_idx, abs_parent_idx] = True
                    attn_mask[abs_parent_idx, abs_node_idx] = True 
            attn_mask[abs_node_idx, abs_node_idx] = True

        full_input_ids = input_ids + dfg_ids
        p_ids = [i + 2 for i in range(c_len)] + [0] * len(dfg_ids)
        padding_len = self.total_len - len(full_input_ids)
        
        if padding_len > 0:
            full_input_ids += [self.tokenizer.pad_token_id] * padding_len
            p_ids += [1] * padding_len
        
        return {
            'input_ids': torch.tensor(full_input_ids, dtype=torch.long),
            'p_ids': torch.tensor(p_ids, dtype=torch.long),
            'attn_mask': torch.tensor(attn_mask, dtype=torch.float),
            'label': torch.tensor(label, dtype=torch.long),
            'index': item
        }

In [4]:
print("Loading dataset...")
tokenizer = AutoTokenizer.from_pretrained("microsoft/graphcodebert-base")
dataset_path = "/kaggle/input/datasets/hasanmahmudabdullah/dfgdataset2/dataset_graphcodebert.jsonl"
full_dataset = TextDataset(tokenizer, args, dataset_path)

# Replicate the 3-Way Split from Test 1 (72/8/20)
n = len(full_dataset)
train_n = int(0.72 * n)
val_n = int(0.08 * n)
test_n = n - train_n - val_n

generator = torch.Generator().manual_seed(args.seed)
_, val_ds, _ = random_split(
    full_dataset, [train_n, val_n, test_n], generator=generator
)

print(f"Validation samples isolated: {len(val_ds)}")

Loading dataset...


config.json:   0%|          | 0.00/539 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

Validation samples isolated: 15996


In [5]:
print("Loading GraphCodeBERT model...")
config = RobertaConfig.from_pretrained("microsoft/graphcodebert-base")
config.num_labels = 2
encoder = RobertaModel.from_pretrained("microsoft/graphcodebert-base", config=config)
model = Model(encoder, config, tokenizer, args)

model_path = "/kaggle/input/datasets/hasanmahmudabdullah/model2/model.bin"
model.load_state_dict(torch.load(model_path, map_location=args.device))
model.to(args.device)
model.eval()
print("Model loaded successfully!")

Loading GraphCodeBERT model...


pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: microsoft/graphcodebert-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Model loaded successfully!


In [6]:
eval_sampler = SequentialSampler(val_ds)
eval_dataloader = DataLoader(val_ds, sampler=eval_sampler, batch_size=args.eval_batch_size)

all_preds = []
all_labels = []
all_probs = []
all_indices = []

print("Running inference on validation set...")
with torch.no_grad():
    for batch in tqdm(eval_dataloader, desc="Evaluating"):
        input_ids = batch['input_ids'].to(args.device)
        p_ids = batch['p_ids'].to(args.device)
        attn_mask = batch['attn_mask'].to(args.device)
        labels = batch['label'].to(args.device)
        indices = batch['index']
        
        probs = model(input_ids=input_ids, p_ids=p_ids, attn_mask=attn_mask)
        preds = torch.argmax(probs, dim=-1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs[:, 1].cpu().numpy())
        all_indices.extend(indices.numpy())

Running inference on validation set...


Evaluating:   0%|          | 0/500 [00:00<?, ?it/s]

In [9]:
false_negatives = []

for i, (pred, label, prob, raw_idx) in enumerate(zip(all_preds, all_labels, all_probs, all_indices)):
    if label == 1 and pred == 0:
        # Get the original JSON line to extract code and metadata
        raw_line = full_dataset.lines[raw_idx]
        raw_data = json.loads(raw_line)
        
        false_negatives.append({
            'index': raw_idx,
            'confidence_safe': 1.0 - prob,
            'code': raw_data.get('code', 'N/A'),
            'project': raw_data.get('filename', 'Unknown')
        })

print(f"\nTotal False Negatives Found in Validation: {len(false_negatives)}")

# Sort by how confident the model was that it was SAFE (worst mistakes first)
false_negatives.sort(key=lambda x: x['confidence_safe'], reverse=True)

print("\n======================================================")
print("Analyzing the Top 10 Most Confident False Negatives:")
print("======================================================\n")

for i, fn in enumerate(false_negatives[:20]):
    print(f"[False Negative #{i+1}] - Originally from '{fn['project']}'")
    print(f"Model Confidence it was SAFE: {fn['confidence_safe'] * 100:.2f}%")
    print("-" * 40)
    lines = fn['code'].split('\n')
    
    # Print up to 35 lines of code
    max_lines = 35
    print('\n'.join(lines[:max_lines]))
    if len(lines) > max_lines:
        print(f"... [Truncated {len(lines) - max_lines} more lines]")
    print("=" * 60 + "\n")


Total False Negatives Found in Validation: 663

Analyzing the Top 10 Most Confident False Negatives:

[False Negative #1] - Originally from 'LVDAndro_58254_file'
Model Confidence it was SAFE: 99.90%
----------------------------------------
public class DummyClass { object2 = new ZipEntry((ZipEntry)object2);
object2.setCompressedSize(-1L);
this.zos.putNextEntry((ZipEntry)object2);
object2 = new byte[(int)this.entry.getSize()];
this.zos.write((byte[])object2);
object2 = zipInputStream2;
object.append(this.toColumn(this.inputCount + this.idealCount));
object.append(object3.toString());
object3 = this.xmlOut;
object3.addAttribute("ref", stringBuilder.toString());
throw new BufferedDataError((Throwable)object); }

[False Negative #2] - Originally from 'LVDAndro_290428_file'
Model Confidence it was SAFE: 99.86%
----------------------------------------
public class DummyClass { public void dummyMethod() { var3 = var4.equals("[]");
var19 = var5.getJSONObject(var1);
var19 = new JSONObject("{\"